<a href="https://colab.research.google.com/github/Pascingo/CTA-media-analysis/blob/main/01_Stage1_CLT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 1: Data Loading, Cleaning and Exploration (Track 1: AI Governance)

In this notebook we prepare the "AI-Media" dataset for our work on Track 1 (AI Governance). The steps are:

1. Setup and download of the dataset from Kaggle
2. Loading and cleaning the data (duplicates, missing values, time frame, text normalization)
3. Filtering the articles that are relevant for AI Governance
4. Corpus statistics
5. Text preprocessing with spaCy (lemmatization, stop words, acronyms)
6. Most frequent words
7. Article volume and sentiment over time
8. Saving the result for Stage 2

# 1. Setup

We put all imports and settings at the top of the notebook. This way we can see directly which libraries are needed, and we can change settings like the time frame or the keywords in one place instead of searching for them in the code. spaCy (with the model `en_core_web_sm`), TextBlob and scikit-learn are already installed in Google Colab.

In [ ]:
import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm

# Settings we use in several cells
TEXT_COLUMN = "content"
START_DATE = "2024-09-01"
END_DATE = "2026-08-31"

GOVERNANCE_KEYWORDS = [
    "security", "accountability", "bias", "fairness",
    "regulatory", "regulation", "governance", "policy",
    "ethics", "compliance", "law"
]

OUTPUT_FILE = "ai_governance_cleaned.csv"
SPACY_CACHE_FILE = "spacy_processed_text.csv"

### Download of the dataset

We import the "AI-Media" dataset directly from Kaggle with the Kaggle API. The login data (`KAGGLE_USERNAME`, `KAGGLE_KEY`) are stored in the Colab secrets, so they are not visible in the notebook.

The download (169 MB) is only done if the CSV file does not exist yet. When we run the notebook again in the same Colab session, this saves time. The file name contains the date of the dataset version (e.g. `ai_media_dataset_20260911.csv`). Instead of writing the name by hand, we search for the file and take the newest one. Like this the notebook also works when a newer version of the dataset is published.

In [ ]:
from google.colab import userdata


os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")

os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
!kaggle datasets download -d jannalipenkova/ai-media-dataset

!unzip -o ai-media-dataset.zip

print("Download and unpacking done.")

We import the "AI-Media"-Dataset directly from Kaggle via the Kaggle API. By importing the dataset this way, we can ensure that it's always actual.

In [ ]:
import pandas as pd
import re

file_name = "ai_media_dataset_20260911.csv"
df = pd.read_csv(file_name)

print(f"Original dataset shape: {df.shape}")
print("Found columns:", df.columns.tolist())


## 2. Loading and cleaning the data

We load the CSV into a pandas DataFrame. The original file has seven columns. The column `Unnamed: 0` is only an old index from the export and has no information, so we do not load it. Our work is about the text of the articles, so the column `content` is our text column (`TEXT_COLUMN`).

In [ ]:
df = pd.read_csv(file_name, usecols=["title", "date", "content", "domain", "url", "tags"])

print(f"Original dataset shape: {df.shape}")
df.head(3)

Now we clean the data in three steps:

1. **Missing values and duplicates:** Articles without text are useless for text analysis, and duplicates would count the same article several times (e.g. in the word frequencies or in the articles per month). We first remove the missing values and then the duplicates.
2. **Time frame:** Our observation period goes from **September 2024 to August 2026**. We filter it here at the beginning, so that all following analyses use the same articles. This also removes the few days of September 2026 in the dataset, which would appear as an almost empty month in the plots.
3. **Text normalization:** We write everything in lowercase, remove HTML tags and all symbols except `. , ! ?`, because these give context about the sentences. Hyphens and slashes are replaced by a space, otherwise words like "london-based" would become "londonbased". At the end we remove double spaces.

For the normalization we use the `.str` functions of pandas instead of a function with `.apply()`. This is shorter and a bit faster.

In [ ]:
# 1. Remove missing texts and duplicates
df = df.dropna(subset=[TEXT_COLUMN])
df = df.drop_duplicates(subset=[TEXT_COLUMN])
print(f"After removing NaN and duplicates: {df.shape}")

# 2. Keep only the articles in our observation period
df["published_at"] = pd.to_datetime(df["date"], errors="coerce")
df = df[(df["published_at"] >= START_DATE) & (df["published_at"] <= END_DATE)].copy()
print(f"After date filter ({START_DATE} to {END_DATE}): {df.shape}")

# 3. Normalize the text
df["cleaned_text"] = (
    df[TEXT_COLUMN]
    .str.lower()                                      # everything in lowercase
    .str.replace(r"<[^>]+>", " ", regex=True)         # remove HTML tags
    .str.replace(r"[-/]", " ", regex=True)            # "london-based" -> "london based"
    .str.replace(r"[^a-z0-9\s.,!?]", "", regex=True)  # remove other symbols, keep .,!?
    .str.replace(r"\s+", " ", regex=True)             # remove double spaces
    .str.strip()
)

df[["title", "cleaned_text"]].head(3)

## 3. Filtering the AI Governance articles

We chose Track 1 (AI Governance). To get the relevant articles, we defined a list of general governance keywords (see setup). The regex uses `\b` (word boundary), so "law" matches "law", but not "lawyer" or "flaw".

For every article we count how many times the keywords appear (`gov_keyword_hits`). All articles with at least one hit are kept in our AI Governance dataset. We keep the number of hits as a column, because we can use it later for a stricter filter.

In [ ]:
pattern = r"\b(?:" + "|".join(GOVERNANCE_KEYWORDS) + r")\b"

df["gov_keyword_hits"] = df["cleaned_text"].str.count(pattern)
df_gov = df[df["gov_keyword_hits"] > 0].copy()

print(f"Size of 'AI Governance' dataset: {df_gov.shape}")
print(f"Share of all articles: {len(df_gov) / len(df):.1%}")
print(f"Articles with at least 3 keyword hits: {(df_gov['gov_keyword_hits'] >= 3).sum()}")

About two thirds of all articles contain at least one keyword. This is a lot, and it shows that the keyword filter is quite broad: words like "security", "policy" or "law" also appear in articles that are not really about AI Governance. For example, the first article in the dataset is a ranking of AI companies in Europe, which only mentions "security" and "regulations" in a few sentences. For Stage 1 we accept this, because we prefer to keep too many articles than to lose relevant ones. With `gov_keyword_hits` we can make the filter stricter later (e.g. at least 3 hits).

## 4. Corpus statistics

To get an overview of the length of the articles, we count the words per article. To be fast, we simply count the spaces (number of words = number of spaces + 1). This works because we removed all double spaces during the cleaning.

In [ ]:
df_gov["n_words"] = df_gov["cleaned_text"].str.count(" ") + 1

audit = pd.DataFrame({
    "documents": [len(df_gov)],
    "min_words": [df_gov["n_words"].min()],
    "median_words": [df_gov["n_words"].median()],
    "max_words": [df_gov["n_words"].max()],
})
display(audit)

The AI Governance dataset contains about 31'600 articles. The shortest article has about 320 words, the median is about 1'300 words and the longest article has more than 9'500 words. So there are no empty or very short texts left, and most articles are long news articles.

This is important for the next stages: transformer models like BERT can only read 512 tokens at once. Since most of our articles are much longer, we will have to split them into parts or shorten them in Stage 2.

## 5. Text preprocessing with spaCy

For the word frequencies and the models in the next stages, we need a normalized version of the texts:

1. **Acronyms and names of laws:** We replace important governance terms with one fixed token, e.g. "eu ai act" becomes `eu_ai_act` and "general data protection regulation" becomes `gdpr`. Otherwise the same law would be split into several separate words. "eu ai act" has to be replaced before "ai act", otherwise only a part of it would be replaced.
2. **Tokenization and lemmatization:** spaCy splits the text into words (tokens) and reduces every word to its base form (lemma), e.g. "regulations" becomes "regulation" and "agents" becomes "agent". Like this, singular and plural are counted together.
3. **Stop words:** We remove stop words ("the", "and", ...), numbers, punctuation and words with less than three letters.

**Runtime:** This is by far the slowest step in the notebook, because spaCy has to analyse about 40 million words. To make it faster we:
- disable the parser and the named entity recognition, because we only need the lemmas,
- use `nlp.pipe()` to process the texts in batches, with `n_process=2` because Colab has two CPU cores,
- save the result in a CSV file (`SPACY_CACHE_FILE`). If we run the cell again, the file is loaded and spaCy is skipped. Note: Colab deletes the files when the runtime is reset. To keep the file longer, it can be saved on Google Drive.

In [ ]:
acronym_map = {
    r"\beu ai act\b": "eu_ai_act",
    r"\bai act\b": "ai_act",
    r"\bgeneral data protection regulation\b": "gdpr",
    r"\bfederal trade commission\b": "ftc",
}
# These tokens contain "_", so they are not "alpha". We keep them anyway.
keep_tokens = set(acronym_map.values())


def lemmatize(doc):
    words = []
    for token in doc:
        if token.text in keep_tokens:
            words.append(token.text)
        elif token.is_alpha and not token.is_stop and len(token.lemma_) > 2:
            words.append(token.lemma_)
    return " ".join(words)


# Try to load the result from an earlier run
if os.path.exists(SPACY_CACHE_FILE):
    cache = pd.read_csv(SPACY_CACHE_FILE, index_col=0)
    if cache.index.equals(df_gov.index):
        df_gov["processed_text"] = cache["processed_text"].fillna("")
        print("Loaded preprocessed texts from cache.")

if "processed_text" not in df_gov.columns:
    # 1. Replace acronyms (order is important: "eu ai act" before "ai act")
    texts = df_gov["cleaned_text"]
    for pattern, replacement in acronym_map.items():
        texts = texts.str.replace(pattern, replacement, regex=True)

    # 2. + 3. Lemmatization and stop words with spaCy
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    docs = nlp.pipe(texts, batch_size=64, n_process=2)
    df_gov["processed_text"] = [lemmatize(doc) for doc in tqdm(docs, total=len(texts))]

    df_gov[["processed_text"]].to_csv(SPACY_CACHE_FILE)
    print(f"Preprocessing done, result saved in '{SPACY_CACHE_FILE}'.")

df_gov[["title", "processed_text"]].head(3)

  0%|          | 0/31667 [00:00<?, ?it/s]

## 6. Most frequent words

To see which topics dominate the dataset, we count the most frequent words. Our first try was a loop with `Counter` over all texts, but it took very long. `CountVectorizer` from scikit-learn does the same much faster, so we only use this version.

We count on `processed_text`, so the stop words are already removed and singular and plural are counted together. In our first try the top 20 were still full of words that appear in almost every article about AI or business ("company", "make", "people", ...). We remove these words with an additional list of domain stop words. Because we count lemmas, the list also contains base forms, e.g. "datum" (spaCy's lemma of "data").

In [ ]:
domain_stopwords = [
    "company", "business", "make", "need", "use", "help", "people", "time",
    "work", "include", "tool", "system", "market", "global", "digital",
    "technology", "artificial", "intelligence", "model", "datum", "data",
    "say", "new", "like", "year"
]

vectorizer = CountVectorizer(stop_words=domain_stopwords, max_features=20)
word_counts = vectorizer.fit_transform(df_gov["processed_text"])

top_words_df = pd.DataFrame({
    "Word": vectorizer.get_feature_names_out(),
    "Frequency": np.asarray(word_counts.sum(axis=0)).ravel(),
}).sort_values("Frequency", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_words_df, x="Frequency", y="Word", color="#D95D4F")
plt.title('Top 20 Words in "AI Governance"')
plt.xlabel("Frequency")
plt.ylabel("Word")
plt.tight_layout()
plt.show()

"Security" is the most frequent word. This fits our topic, but "security" can mean cybersecurity as well as AI safety, so we cannot say from the count alone which meaning dominates. Most of the other top words ("platform", "service", "software", "cloud", "agent", "management", ...) are about technology and business rather than governance. This confirms what we saw in section 3: the keyword filter is broad, and many articles in the dataset are business or product news that only mention governance topics briefly. Specific governance terms like "regulation" or `eu_ai_act` are not in the top 20.




## 7. Article volume and sentiment over time

Finally we look at how the reporting on AI Governance changes over time:

1. **Volume:** number of articles per month, to see when the topic got more attention.
2. **Sentiment:** With TextBlob we calculate the polarity of every article, from -1 (negative) to +1 (positive), and then the average per month. This shows if the tone of the reporting changes.

We show both in one figure, so that peaks in the volume can be compared directly with changes in the sentiment. The sentiment calculation takes a few minutes, so we show a progress bar.

In [ ]:
df_gov["sentiment_polarity"] = [
    TextBlob(text).sentiment.polarity for text in tqdm(df_gov["cleaned_text"])
]

monthly = df_gov.groupby(df_gov["published_at"].dt.to_period("M")).agg(
    article_count=("cleaned_text", "count"),
    avg_sentiment=("sentiment_polarity", "mean"),
)
monthly.index = monthly.index.astype(str)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Articles per month
ax1.bar(monthly.index, monthly["article_count"], color="#3D7C83")
ax1.set_title("AI Governance: Articles per Month (Sep 2024 - Aug 2026)")
ax1.set_ylabel("Number of articles")

# Average sentiment per month
ax2.plot(monthly.index, monthly["avg_sentiment"], marker="o", color="#D95D4F", linewidth=2)
ax2.axhline(0, color="grey", linestyle="--", linewidth=1)
ax2.set_title("AI Governance: Average Sentiment Polarity per Month")
ax2.set_xlabel("Month")
ax2.set_ylabel("Polarity (-1 to +1)")
ax2.grid(True, linestyle="--", alpha=0.6)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Volume:** There is regular reporting on AI Governance over the whole period, with about 900 to 1'600 articles in most months. September 2024 has fewer articles because the dataset only starts in the middle of the month. There are two clear peaks in January 2026 (about 2'100 articles) and May 2026 (about 2'400 articles). One possible reason is the release of new frontier models and the discussion about their capabilities and risks. We will check this in the next stage by looking at the topics of these months. August 2026 has clearly fewer articles; this could be because the data for this month is not complete yet.

**Sentiment:**

## 8. Saving the result

We save the AI Governance dataset as a CSV file, so that the whole team can start Stage 2 directly with it and does not have to run the slow steps again. We keep the original text (`content`) as well, because some methods in the next stages (e.g. named entity recognition) work better with the original upper and lower case.

In [ ]:
columns_to_save = [
    "title", "date", "domain", "url", "tags", "content",
    "cleaned_text", "processed_text", "gov_keyword_hits", "n_words", "sentiment_polarity"
]
df_gov[columns_to_save].to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(df_gov)} articles in '{OUTPUT_FILE}'.")